# Stage 1 — Wikidata query collection

**Pipeline:** **01_query_collection** → 02_triplet_mining → rebuild_wikidata_clean_splits.py (see `datasets/DATASET.md`).

**Purpose:** fetch queries from the LSQ Wikidata log, execute each on the Wikidata endpoint and keep those that return at least one row.

**Run from:** `datasets/scripts/`

**Outputs:**
- `../wikidata-splits/sources/lsq_sample_wikidata.csv` — all fetched LSQ query texts
- `../wikidata-splits/sources/successful_queries_wikidata.json` — non-empty queries, input of Stage 2

**Note:** LSQ is paged with `LIMIT`/`OFFSET` and no `ORDER BY`, so a rerun is not guaranteed to return the same queries. Non-SELECT queries are not filtered here; the rebuild drops them (status `unsupported_non_select`). The released dataset is fixed by `../wikidata-splits/unique_queries.json` (see `DATASET.md`).


In [ ]:
import os
import time
import json
import pandas as pd
import requests
from tqdm import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [ ]:

# ---------- CONFIG ----------
LSQ_ENDPOINT = "http://lsq.aksw.org/sparql"
BATCH_SIZE = 1000
NUM_BATCHES = 300           # reduce for quick sampled runs
SAMPLE_SIZE = None         # set to int to cap total queries after fetch (None = no cap)
from pathlib import Path
SOURCES = Path("../wikidata-splits/sources")
SOURCES.mkdir(parents=True, exist_ok=True)
LSQ_SAMPLE_CSV = SOURCES / "lsq_sample_wikidata.csv"
OUTPUT_FILE = SOURCES / "successful_queries_wikidata.json"
MIN_RESULTS_PER_QUERY = 1
MAX_BINDINGS_PER_QUERY = 50
REQUEST_TIMEOUT = 15
DELAY_BETWEEN_REQUESTS = 0.1
SAVE_EVERY = 50


In [ ]:
WIKIDATA_ENDPOINT = "https://wikidata.data.dice-research.org/sparql"
QUERY_TEMPLATE_WIKIDATA = """
PREFIX lsqv: <http://lsq.aksw.org/vocab#>
PREFIX sd:   <http://www.w3.org/ns/sparql-service-description#>

SELECT DISTINCT ?text
FROM <http://lsq.aksw.org/wikidata>
WHERE {{
  ?query lsqv:text ?text .
  ?query lsqv:hasSpin ?spin .
  ?spin a ?spinType .
  ?query lsqv:hasStructuralFeatures ?features .
}}
LIMIT {limit}
OFFSET {offset}
"""

def fetch_select_queries_lsqs(
    num_batches=NUM_BATCHES,
    batch_size=BATCH_SIZE,
    sample_size=SAMPLE_SIZE,
    delay=0.5,
    timeout=120
):
    all_queries = []

    for i in range(num_batches):
        offset = i * batch_size
        print(f"Fetching batch {i+1}/{num_batches}, OFFSET {offset}")

        try:
            sparql = SPARQLWrapper(LSQ_ENDPOINT)
            sparql.setQuery(
                QUERY_TEMPLATE_WIKIDATA.format(
                    limit=batch_size,
                    offset=offset
                )
            )
            sparql.setReturnFormat(JSON)
            sparql.setTimeout(timeout)

            results = sparql.query().convert()
            texts = [
                b["text"]["value"]
                for b in results.get("results", {}).get("bindings", [])
            ]

            if not texts:
                print("No results returned — stopping early.")
                break

            all_queries.extend(texts)
            print(f"Fetched {len(texts)} queries")

        except Exception as e:
            print(f"LSQ fetch error at OFFSET {offset}: {e}")

        time.sleep(delay)

    if sample_size:
        all_queries = all_queries[:sample_size]

    return pd.DataFrame({"query": all_queries})

In [ ]:
# Uncomment to run fetch
df = fetch_select_queries_lsqs()
df.to_csv(LSQ_SAMPLE_CSV, index=False)

In [ ]:
# ---------- Load sampled queries ----------
if not os.path.exists(LSQ_SAMPLE_CSV):
    raise FileNotFoundError("Run the LSQ fetch step first and save it")

df = pd.read_csv(LSQ_SAMPLE_CSV)
print(f"Loaded {len(df)} sampled queries")
df_unique = df.drop_duplicates(subset="query").reset_index(drop=True)
print(f"Unique queries after dedup: {len(df_unique)}")


In [ ]:
# ---------- Prepare HTTP session with retries ----------
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=(429,500,502,503,504))
session.mount("https://", HTTPAdapter(max_retries=retries))
HEADERS = {"Accept": "application/sparql-results+json"}

def query_dbpedia(q):
    try:
        r = session.get(WIKIDATA_ENDPOINT, params={"query": q}, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"_error": str(e)}

In [ ]:
# ---------- Run queries and collect successful ones (sample-friendly) ----------
successful = []
for i, row in tqdm(df_unique.iterrows(), total=len(df_unique), desc="Querying Wikidata"):
    sparql = str(row["query"]).strip()
    if not sparql:
        continue
    res = query_dbpedia(sparql)
    if "_error" in res:
        # skip / optionally log
        continue
    bindings = res.get("results", {}).get("bindings", []) or []
    if len(bindings) >= MIN_RESULTS_PER_QUERY:
        successful.append({
            "sparql_query": sparql,
            "results_count": len(bindings),
            "results": bindings[:MAX_BINDINGS_PER_QUERY]
        })
    time.sleep(DELAY_BETWEEN_REQUESTS)
    if SAVE_EVERY and len(successful) and len(successful) % SAVE_EVERY == 0:
        with open(str(OUTPUT_FILE) + ".partial", "w", encoding="utf-8") as f:
            json.dump(successful, f, ensure_ascii=False, indent=2)


In [ ]:
# ---------- Final save ----------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(successful, f, ensure_ascii=False, indent=2)

print(f"Saved {len(successful)} successful queries to {OUTPUT_FILE}")